## Install Dependencies

In [ ]:
!pip install -U ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 139.0 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.90
    Uninstalling opencv-python-headless-4.13.0.90:
      Successfully uninstalled opencv-python-headless-4.13.0.90
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


## Create Project Folder in Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/FYP_YOLO_Hand_Detection"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)

print("Drive project folder ready:", DRIVE_PROJECT_DIR)

## Download Dataset from Roboflow (API)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="enter-your-roboflow-api-key-here")
project = rf.workspace("objectdetection-3awm0").project("hand_sign_language")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to hand_sign_language-1 in yolov8:: 100%|██████████| 1679/1679 [00:00<00:00, 4084.27it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Verify Dataset Structure

In [ ]:
!ls hand_sign_language-1

data.yaml  README.roboflow.txt	test  train  valid


## Load YOLOv8 Model

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

## Train YOLOv8 - 50 epochs

In [ ]:
model.train(
    data="hand_sign_language-1/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="yolo_hand_detection",
    name="phase1_hand_detector"
)

Ultralytics 8.4.13 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=hand_sign_language-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=phase1_hand_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience

## Evaluate Model

In [ ]:
model.val(
    data="hand_sign_language-1/data.yaml",
    split="test"
)

## Run Inference on Test Images

In [ ]:
model.predict(
    source="hand_sign_language-1/test/images",
    conf=0.4,
    save=True
)

## Save Trained Model

In [ ]:
model.export(format="pt")

## Copy ALL YOLO Outputs to Drive

In [ ]:
import shutil

SRC_RUNS = "runs"
DST_RUNS = os.path.join(DRIVE_PROJECT_DIR, "runs_backup")

if os.path.exists(DST_RUNS):
    shutil.rmtree(DST_RUNS)

shutil.copytree(SRC_RUNS, DST_RUNS)

print("✅ All YOLO runs backed up to Google Drive!")

## Copy Trained Weights Separately

In [ ]:
WEIGHTS_SRC = "runs/detect/yolo_hand_detection/phase1_hand_detector/weights"
WEIGHTS_DST = os.path.join(DRIVE_PROJECT_DIR, "weights")

os.makedirs(WEIGHTS_DST, exist_ok=True)

for file in os.listdir(WEIGHTS_SRC):
    shutil.copy(
        os.path.join(WEIGHTS_SRC, file),
        WEIGHTS_DST
    )

print("✅ Model weights saved separately!")

## Save Dataset Metadata

In [ ]:
shutil.make_archive(
    os.path.join(DRIVE_PROJECT_DIR, "dataset_yolo"),
    'zip',
    "hand_sign_language-1"
)

print("✅ Dataset zipped and saved!")